# R4: Attribute checks

**Goal:** check and fix the non-spatial values: sowing and harvest dates, crop duration, variety
names, categories, soil values and missing data. Every rule comes from a statistical test or a
published source, and every change gets a flag. Input: `data/processed_v2/R3_geometry.gpkg`.

## Step 1: Sowing dates

Two methods. (1) **Agronomic window:** 1 June (start of the monsoon season) to 5 August, the latest
sowing tested in Maharashtra field trials (Kathmale et al.). A sowing outside it is implausible.
A wrong year is fixed only if the day and month fall inside the window.
(2) **Statistical check per year:** a Kruskal-Wallis test shows whether the normal sowing date differs
between years. If it does, dates outside Q1 - 3 IQR to Q3 + 3 IQR of their own year are flagged
as unusual (kept, not removed). A Mann-Whitney U test compares the fixed dates with the rest of their year.

**Found:** 17 sowing dates had the wrong year; 15 were fixed (day and month normal). 5 dates are
implausible (20 May, 14 Aug, 2 Nov, and the 1960 and 1964 dates) and are not used. 34 are missing.
The normal sowing date differs strongly between years (Kruskal-Wallis H = 302, p < 0.001):
median 28 June (2022), 5 July (2023), 19 June (2024), 18 June (2025).
So each year is checked against its own range: 43 dates are unusual for their year
(3 in 2023, 4 in 2024, 36 in 2025), flagged and kept. The 15 corrected dates are consistent with the
rest of their year (Mann-Whitney U, p = 0.30: no evidence of a difference). This is only a weak check,
because we fixed only dates whose day and month were already normal.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
DATA2 = ROOT / "data" / "processed_v2"
OUT = ROOT / "results" / "phase1a"

f = gpd.read_file(DATA2 / "R3_geometry.gpkg")
print("fields:", len(f))

# sowing date is written day/month/year
sow = pd.to_datetime(f["sowing_date_raw"].astype(str).str.strip(), format="%d/%m/%Y", errors="coerce")
md = sow.dt.month * 100 + sow.dt.day                     # month-day as a number, e.g. 615 = 15 June

# (1) agronomic window: 1 June (start of the monsoon season) to 5 August (latest sowing tested
#     in Maharashtra field trials, Kathmale et al.). Outside it = not a plausible kharif sowing.
in_window = md.between(601, 805)
right_year = sow.dt.year == f["year"]

# wrong year: a typo only if day and month are inside the window -> fix the year
fix_year = sow.notna() & ~right_year & in_window
f["sowing_date"] = sow.where(right_year & in_window)
f.loc[fix_year, "sowing_date"] = [pd.Timestamp(year=int(y), month=t.month, day=t.day)
                                  for y, t in zip(f.loc[fix_year, "year"], sow[fix_year])]
f["sowing_doy"] = f["sowing_date"].dt.dayofyear
f["flag_sowing_year_fixed"] = fix_year
f["flag_sowing_implausible"] = sow.notna() & ~in_window
f["flag_sowing_missing"] = sow.isna()

print(f"\nwrong year: {int((sow.notna() & ~right_year).sum())} | fixed (day and month inside the window): {int(fix_year.sum())}")
print("outside the agronomic window (1 Jun to 5 Aug):", int(f["flag_sowing_implausible"].sum()))
print("  ", f.loc[f["flag_sowing_implausible"], ["year", "sowing_date_raw"]].values.tolist())
print("missing:", int(f["flag_sowing_missing"].sum()))

# (2) does the normal sowing date differ between years? (Kruskal-Wallis)
groups = [f.loc[f["year"] == y, "sowing_doy"].dropna() for y in sorted(f["year"].unique())]
kw = stats.kruskal(*groups)
print(f"\nsowing day differs between years: Kruskal-Wallis H = {kw.statistic:.0f}, p = {kw.pvalue:.1e}")

# (3) so the statistical check is done per year: outside Q1 - 3 IQR to Q3 + 3 IQR of that year = unusual
rows = []
f["flag_sowing_unusual"] = False
for y, g in f.groupby("year"):
    d = g["sowing_doy"].dropna()
    q1, q3 = d.quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    unusual = g["sowing_doy"].notna() & ~g["sowing_doy"].between(lo, hi)
    f.loc[unusual[unusual].index, "flag_sowing_unusual"] = True
    day = lambda n: (pd.Timestamp(int(y), 1, 1) + pd.Timedelta(days=int(n) - 1)).strftime("%d %b")
    rows.append({"year": y, "n": len(d), "median": day(d.median()), "Q1": day(q1), "Q3": day(q3),
                 "fence_low": day(max(lo, 1)), "fence_high": day(hi), "unusual": int(unusual.sum())})
per_year = pd.DataFrame(rows)
per_year.to_csv(OUT / "R4_sowing_by_year.csv", index=False)
print(per_year.to_string(index=False))

# check: do the fixed-year dates look like the others of their year? (Mann-Whitney U)
fixed = f.loc[fix_year, "sowing_doy"]; others = f.loc[~fix_year & f["sowing_doy"].notna() & f["year"].isin(f.loc[fix_year, "year"]), "sowing_doy"]
print(f"\nfixed dates vs the rest of their years: Mann-Whitney U p = {stats.mannwhitneyu(fixed, others).pvalue:.2f}")

fields: 1207

wrong year: 17 | fixed (day and month inside the window): 15
outside the agronomic window (1 Jun to 5 Aug): 5
   [[2023, '02/11/2023'], [2023, '14/08/2023'], [2025, '20/05/2025'], [2025, '23/08/1964'], [2025, '15/09/1960']]
missing: 34

sowing day differs between years: Kruskal-Wallis H = 302, p = 4.3e-65
 year   n median     Q1     Q3 fence_low fence_high  unusual
 2022  38 28 Jun 22 Jun 01 Jul    26 May     28 Jul        0
 2023  58 05 Jul 04 Jul 07 Jul    25 Jun     16 Jul        3
 2024 306 19 Jun 16 Jun 20 Jun    04 Jun     02 Jul        4
 2025 766 18 Jun 15 Jun 19 Jun    03 Jun     01 Jul       36

fixed dates vs the rest of their years: Mann-Whitney U p = 0.30


## Step 2: Harvest dates and the day/month swap

A date like 20/10 can only be read one way (clear). A date like 11/10 could be 10 Nov or 11 Oct (unclear).
(1) **Is there a swap?** The clear dates give the normal harvest season (kernel density per year).
We count it only on days 1 to 12 of each month, because unclear dates always have a day of 1 to 12.
A chi-square test compares the months of the unclear dates (as written) with this expectation.
(2) **Which reading is right?** For each unclear date we compute the crop duration (harvest minus sowing)
for both readings and check how common each duration is among the clear dates. A reading is chosen only
if it is at least 3 times as likely as the other ("positive evidence", Kass and Raftery, 1995).
Otherwise the date is undecided and flagged. Without a sowing date, the harvest day of year is used.
(3) The chi-square test is repeated after the fix.

**Found:** 691 dates are clear and 516 unclear. Read as written, the unclear dates fall too rarely in
October (56% vs 81% expected) and too often in other months (8% vs 1%); chi-square p < 0.001, so the
swap is real. 85 dates were swapped, 409 kept as written, and 22 are undecided (flagged, not guessed).
After the fix, November (13% vs 14%) and other months (1% vs 1%) match the expectation and October
rises to 73%. The fix is only partial: the chi-square test is still significant after it (p < 0.001),
because September is still too high (12% vs 4% expected). These are 61 early-September harvests; for 55
of them the swapped reading is impossible, so they are not swaps. Their crop duration is short
(median 77 days) and is checked in Step 4. 3 harvest dates had the wrong year; the crop year was used.

In [2]:
# harvest date is written month/day/year. R1 showed some have day and month swapped.
# A date like 20/10 has only one reading ("clear"); a date like 11/10 has two ("unclear").
p = (f["harvest_date_raw"].astype(str).str.strip().str.split(" ").str[0]
     .str.split("/", expand=True).apply(pd.to_numeric, errors="coerce"))
a, b, yr = p[0], p[1], p[2]
cy = f["year"]                                                   # crop year
make = lambda m, d: pd.to_datetime(dict(year=cy, month=m, day=d), errors="coerce")
read_md, read_dm = make(a, b), make(b, a)                        # as written, and swapped
clear = (b > 12) | (a == b)
unclear = ~clear & read_md.notna() & read_dm.notna()
f["flag_harvest_year_fixed"] = yr != cy                          # year typo: crop year used instead

# fair reference for the month check: the season curve of the clear dates (kernel density, per year),
# counted only on days 1 to 12 of each month, because unclear dates always have a day of 1 to 12
def month_mix(d):
    m = d.dt.month.where(d.dt.month.isin([9, 10, 11]), 0)
    return m.value_counts().reindex([9, 10, 11, 0], fill_value=0)
def expected_mix(mask):
    tot = pd.Series(0.0, index=[9, 10, 11, 0])
    for y in sorted(cy.unique()):
        k = stats.gaussian_kde(read_md[clear & (cy == y)].dt.dayofyear)
        days = pd.date_range(f"{y}-01-01", f"{y}-12-31"); days = days[days.day <= 12]
        w = pd.Series(k(days.dayofyear.to_numpy()), index=days.month); w = w / w.sum()
        m = w.groupby(level=0).sum()
        share = pd.Series({9: m.get(9, 0), 10: m.get(10, 0), 11: m.get(11, 0)}); share[0] = 1 - share.sum()
        tot += share * int((mask & (cy == y)).sum())
    return tot
pct = lambda m: (100 * m / m.sum()).round(1).tolist()

# (1) is there a swap at all? (chi-square goodness of fit)
obs = month_mix(read_md[unclear]); exp = expected_mix(unclear)
print("harvest month share   [Sep, Oct, Nov, other] %")
print("  expected from the clear dates:", pct(exp))
print("  unclear dates, as written:    ", pct(obs), f"| chi-square p = {stats.chisquare(obs, exp / exp.sum() * obs.sum()).pvalue:.1e}")

# (2) which reading is right? How common is each reading's crop duration among the clear dates?
dur_md = (read_md - f["sowing_date"]).dt.days
dur_dm = (read_dm - f["sowing_date"]).dt.days
ref = dur_md[clear & f["sowing_date"].notna()]
q1, q3 = ref.quantile([0.25, 0.75]); lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
kde = stats.gaussian_kde(ref[ref.between(lo, hi)])
like = lambda d: pd.Series(np.where(d.between(lo, hi), kde(d.fillna(0).to_numpy()), 0.0), index=f.index)
L_md, L_dm = like(dur_md), like(dur_dm)
nosow = f["sowing_date"].isna()                     # no sowing date: use the harvest day of year instead
for y in sorted(cy.unique()):
    k = stats.gaussian_kde(read_md[clear & (cy == y)].dt.dayofyear)
    r = nosow & (cy == y)
    L_md[r] = k(read_md[r].dt.dayofyear.fillna(0).to_numpy())
    L_dm[r] = k(read_dm[r].dt.dayofyear.fillna(0).to_numpy())

BF = 3   # one reading must be at least 3 times as likely: "positive evidence" (Kass and Raftery, 1995)
choose_dm = unclear & (L_dm > BF * L_md)
keep_md = unclear & (L_md > BF * L_dm)
undecided = unclear & ~(choose_dm | keep_md)
f["harvest_date"] = read_md.where(clear | keep_md)
f.loc[choose_dm, "harvest_date"] = read_dm[choose_dm]
f["flag_harvest_swapped"] = choose_dm
f["flag_harvest_undecided"] = undecided
print(f"\nclear: {int(clear.sum())} | unclear: {int(unclear.sum())} -> kept as written {int(keep_md.sum())}, "
      f"swapped {int(choose_dm.sum())}, undecided {int(undecided.sum())} | year typos fixed: {int(f['flag_harvest_year_fixed'].sum())}")

# (3) after the fix
done = unclear & f["harvest_date"].notna()
obs2 = month_mix(f.loc[done, "harvest_date"]); exp2 = expected_mix(done)
print("  unclear dates, after the fix: ", pct(obs2), f"| chi-square p = {stats.chisquare(obs2, exp2 / exp2.sum() * obs2.sum()).pvalue:.1e}")
early_sep = done & (f["harvest_date"].dt.month == 9)
print("  September dates left:", int(early_sep.sum()), "| their swapped reading is impossible:",
      int((early_sep & (L_dm == 0)).sum()), "| median crop duration:", int(dur_md[early_sep].median()), "days")

harvest month share   [Sep, Oct, Nov, other] %
  expected from the clear dates: [3.9, 81.3, 13.9, 0.9]
  unclear dates, as written:     [16.3, 56.2, 19.4, 8.1] | chi-square p = 6.3e-124

clear: 691 | unclear: 516 -> kept as written 409, swapped 85, undecided 22 | year typos fixed: 3
  unclear dates, after the fix:  [12.3, 73.3, 13.0, 1.4] | chi-square p = 5.7e-21
  September dates left: 61 | their swapped reading is impossible: 55 | median crop duration: 77 days


## Step 3: Variety names

The same variety is written many ways ("KDS 753", "kds753", "753", Marathi "केडीएस ७५३").
Every released soybean variety has a code number, so the rule matches on that number:
(1) Marathi letters and digits are turned into English ones;
(2) exactly one number that is a known variety code -> that variety ("exact code");
(3) a number one digit away from exactly one known code -> that variety ("one-digit typo", checked by hand);
(4) no number, two numbers (a mix) or no close code -> "Unknown". Nothing is guessed.
The full lookup table (raw spelling, standard name, rule used, fields) is saved for checking.

**Found:** 73 raw spellings become 11 varieties. 1,158 fields match an exact code; 4 are one-digit
typos (754 and 752 -> KDS 753, 218 -> DS 228). 45 fields are Unknown (34 blank, 11 junk or mixed
values such as "late", "kds500" or two varieties in one cell). KDS 753 (471) and DS 228 (294) make up
63% of fields. Three varieties are rare (MAUS 71: 9, MAUS 162: 7, MAUS 158: 3); see Step 5.

In [3]:
import re

# released soybean varieties seen in the data: code number -> standard name
RELEASED = {"753": "KDS 753", "228": "DS 228", "992": "KDS 992", "335": "JS 335", "612": "MAUS 612",
            "726": "KDS 726", "725": "KDS 725", "71": "MAUS 71", "162": "MAUS 162", "158": "MAUS 158",
            "1001": "AMS 1001"}

DEVANAGARI = str.maketrans("०१२३४५६७८९", "0123456789")
WORDS = {"केडीएस": "KDS", "डीएस": "DS", "एमएयुएस": "MAUS", "किमया": ""}

def edit_distance(s, t):
    """Number of single-character changes to turn s into t (Levenshtein)."""
    d = list(range(len(t) + 1))
    for i, cs in enumerate(s, 1):
        prev, d[0] = d[0], i
        for j, ct in enumerate(t, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (cs != ct))
    return d[-1]

def match(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return pd.Series({"variety": "Unknown", "how": "missing"})
    s = str(raw).translate(DEVANAGARI)
    for k, v in WORDS.items():
        s = s.replace(k, v)
    nums = re.findall(r"\d+", s.replace(".", " "))
    if len(nums) != 1:
        return pd.Series({"variety": "Unknown", "how": "no number" if not nums else "two or more numbers"})
    n = nums[0]
    if n in RELEASED:
        return pd.Series({"variety": RELEASED[n], "how": "exact code"})
    close = [k for k in RELEASED if len(k) >= 3 and edit_distance(n, k) == 1]
    if len(close) == 1:
        return pd.Series({"variety": RELEASED[close[0]], "how": "one-digit typo"})
    return pd.Series({"variety": "Unknown", "how": "no match"})

m = f["variety_raw"].apply(match)
f["variety"], f["variety_match"] = m["variety"], m["how"]

lookup = (pd.DataFrame({"raw": f["variety_raw"].fillna("(blank)"), "variety": f["variety"], "how": f["variety_match"]})
          .value_counts().reset_index(name="fields").sort_values(["variety", "fields"], ascending=[True, False]))
lookup.to_csv(OUT / "R4_variety_lookup.csv", index=False)

print("raw spellings:", f["variety_raw"].nunique(), "-> standard varieties:", f.loc[f["variety"] != "Unknown", "variety"].nunique())
print("\nhow each field was matched:")
print(f["variety_match"].value_counts().to_string())
print("\nfields per variety:")
print(f["variety"].value_counts().to_string())
print("\none-digit typos (check by hand):")
print(lookup[lookup["how"] == "one-digit typo"].to_string(index=False))
print("\nnot matched (Unknown):")
print(lookup[(lookup["variety"] == "Unknown") & (lookup["how"] != "missing")].to_string(index=False))

raw spellings: 73 -> standard varieties: 11

how each field was matched:
variety_match
exact code             1158
missing                  34
no match                  7
one-digit typo            4
two or more numbers       3
no number                 1

fields per variety:
variety
KDS 753     471
DS 228      294
KDS 992     106
JS 335       99
MAUS 612     94
KDS 726      49
Unknown      45
KDS 725      18
AMS 1001     12
MAUS 71       9
MAUS 162      7
MAUS 158      3

one-digit typos (check by hand):
raw variety            how  fields
218  DS 228 one-digit typo       1
754 KDS 753 one-digit typo       2
752 KDS 753 one-digit typo       1

not matched (Unknown):
           raw variety                 how  fields
          3344 Unknown            no match       1
        kds500 Unknown            no match       1
KDS-726,DS-228 Unknown two or more numbers       1
          late Unknown           no number       1
             2 Unknown            no match       1
          1208 Unkno

## Step 4: Crop duration (harvest minus sowing)

(1) **What does duration depend on?** Kruskal-Wallis tests whether it differs between varieties
and between years; the check is then done on the factor that matters.
(2) **Statistical check per year:** outside Q1 - 3 IQR to Q3 + 3 IQR of its own year = unusual.
(3) **Agronomic check:** harvest before 91 days is earlier than the maturity of any listed variety
(earliest: MAUS 612, 91 to 95 days; ICAR oilseeds compendium), so the crop could not have matured.
Fields are flagged, not removed.

**Found:** 1,146 fields have both dates; median crop duration 114 days (Q1 106, Q3 123).
Duration does not differ significantly between varieties (Kruskal-Wallis p = 0.07) but does between
years (p < 0.001; 2023 is shortest at 103 days, after late sowing), so the statistical check is per year.
It flags only 6 fields (2 too short, 4 too long). The agronomic check flags 91 fields harvested before
91 days, mostly in September (75). The two methods disagree because these 91 are a group (8% of fields),
not rare outliers, so only the published maturity shows they are too short. Both flags are kept;
Phase 1B tests whether these fields differ in yield. Note: "KDS 725" (18 fields, 2025 only) is not a
released variety we could find; it may be MAUS 725, recommended for Maharashtra in 2025. Kept as written.

In [4]:
f["crop_duration"] = (f["harvest_date"] - f["sowing_date"]).dt.days
d = f.dropna(subset=["crop_duration"])
print(f"fields with both dates: {len(d)} | crop duration median {d['crop_duration'].median():.0f} days "
      f"(Q1 {d['crop_duration'].quantile(.25):.0f}, Q3 {d['crop_duration'].quantile(.75):.0f})")

# (1) does the crop duration depend on the variety, or on the year? (Kruskal-Wallis)
by_var = [g["crop_duration"] for v, g in d[d["variety"] != "Unknown"].groupby("variety") if len(g) >= 10]
by_year = [g["crop_duration"] for y, g in d.groupby("year")]
kv, ky = stats.kruskal(*by_var), stats.kruskal(*by_year)
print(f"differs between varieties (10+ fields each): H = {kv.statistic:.1f}, p = {kv.pvalue:.2f}")
print(f"differs between years:                        H = {ky.statistic:.1f}, p = {ky.pvalue:.1e}")

# (2) statistical check per year: outside Q1 - 3 IQR to Q3 + 3 IQR of its own year = unusual
rows = []
f["flag_duration_unusual"] = False
for y, g in d.groupby("year"):
    q1, q3 = g["crop_duration"].quantile([0.25, 0.75]); lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    out = g.index[~g["crop_duration"].between(lo, hi)]
    f.loc[out, "flag_duration_unusual"] = True
    rows.append({"year": y, "n": len(g), "median": g["crop_duration"].median(), "fence_low": lo, "fence_high": hi,
                 "too_short": int((g["crop_duration"] < lo).sum()), "too_long": int((g["crop_duration"] > hi).sum())})
per_year = pd.DataFrame(rows).round(0)
per_year.to_csv(OUT / "R4_duration_by_year.csv", index=False)
print(per_year.to_string(index=False))

# (3) agronomic check: harvest before the earliest maturity of the listed varieties
#     (MAUS 612: 91 to 95 days, ICAR oilseeds compendium) -> harvested before any variety can mature
EARLIEST_MATURITY = 91
f["flag_harvest_before_maturity"] = f["crop_duration"] < EARLIEST_MATURITY
print(f"\nharvested before {EARLIEST_MATURITY} days (before any listed variety matures): {int(f['flag_harvest_before_maturity'].sum())}")
print("  of these, also unusual for their year:", int((f["flag_harvest_before_maturity"] & f["flag_duration_unusual"]).sum()))
print("  by harvest month:", f.loc[f["flag_harvest_before_maturity"], "harvest_date"].dt.month.value_counts().sort_index().to_dict())

fields with both dates: 1146 | crop duration median 114 days (Q1 106, Q3 123)
differs between varieties (10+ fields each): H = 13.2, p = 0.07
differs between years:                        H = 62.9, p = 1.4e-13
 year   n  median  fence_low  fence_high  too_short  too_long
 2022  36   122.0       56.0       188.0          0         0
 2023  55   103.0       61.0       142.0          2         0
 2024 300   114.0       45.0       185.0          0         3
 2025 755   115.0       63.0       168.0          0         1

harvested before 91 days (before any listed variety matures): 91
  of these, also unusual for their year: 2
  by harvest month: {6: 1, 8: 7, 9: 75, 10: 8}


## Step 5: Categories and rare groups

Rule: a group needs at least 10 fields. With fewer, its effect cannot be estimated, and statistical
tests on categories become unreliable (a chi-square test needs an expected count of at least 5 per
cell, Cochran 1954; 10 keeps that true even when the data is split in two). Rare groups are merged
into the nearest group by meaning (for ordered categories, the next step up), or into "Other".
The original column is kept; the merged one is saved as `<column>_group`. "Slit" is corrected to "Silt".

**Found:** irrigation, soil texture and soil depth need no change (smallest groups: Drip 219, Sand 28,
Shallow 11). Soil colour: White (2) and Red (1) join Brown as "Brown or lighter" (51). Soil structure:
Coarse (6) joins Medium as "Medium or coarse" (111). Variety: MAUS 71 (9), MAUS 162 (7) and MAUS 158 (3)
become "Other variety" (19), leaving 10 groups, the smallest AMS 1001 (12). "Slit" is now "Silt".

In [5]:
MIN_N = 10   # fewest fields a group needs (roadmap rule; keeps chi-square expected counts >= 5, Cochran 1954)

# spelling fix first
f["soil_texture"] = f["soil_texture"].replace({"Slit": "Silt"})

# rare groups are merged into the nearest group by meaning, or into "Other"
MERGE = {
    "soil_color": {"White": "Brown or lighter", "Red": "Brown or lighter", "Brown": "Brown or lighter"},  # the lighter soils together
    "soil_structure": {"Coarse": "Medium or coarse", "Medium": "Medium or coarse"},   # ordered: next size up
    "variety": {"MAUS 71": "Other variety", "MAUS 162": "Other variety", "MAUS 158": "Other variety"},
}
rows = []
for col in ["irrigation", "soil_color", "soil_structure", "soil_texture", "soil_depth", "variety"]:
    before = f[col].value_counts()
    rare = before[before < MIN_N]
    f[col + "_group"] = f[col].replace(MERGE.get(col, {}))
    after = f[col + "_group"].value_counts()
    rows.append({"column": col, "groups_before": len(before), "rare (< 10 fields)": ", ".join(f"{k} ({v})" for k, v in rare.items()) or "-",
                 "groups_after": len(after), "smallest_after": f"{after.idxmin()} ({after.min()})"})
cats = pd.DataFrame(rows)
cats.to_csv(OUT / "R4_categories.csv", index=False)
print(cats.to_string(index=False))
print("\nsoil texture after the spelling fix:", f["soil_texture"].value_counts().to_dict())

        column  groups_before                      rare (< 10 fields)  groups_after         smallest_after
    irrigation              3                                       -             3             Drip (219)
    soil_color              5                      White (2), Red (1)             3  Brown or lighter (51)
soil_structure              4                              Coarse (6)             3 Medium or coarse (111)
  soil_texture              5                                       -             5              Sand (28)
    soil_depth              3                                       -             3           Shallow (11)
       variety             12 MAUS 71 (9), MAUS 162 (7), MAUS 158 (3)            10          AMS 1001 (12)

soil texture after the spelling fix: {'Black Cotton Soil / Clay': 474, 'Clay Loam': 219, 'Silt': 209, 'Loam': 44, 'Sand': 28}


## Step 6: Soil lab values

(1) A value of exactly 0 is not a real measurement (soil EC cannot be 0), so it becomes a blank.
(2) Coverage: how many fields have all six lab values, in which years and villages.
(3) Are the values sensible? Each one is compared with the Soil Health Card limits (ICAR-IISS):
organic carbon % (low < 0.5, high > 0.75); available N, P, K in kg/ha (low < 280 / 10 / 120,
high > 560 / 25 / 280); pH (acidic < 6.5, neutral 6.5 to 7.0, alkaline > 7.0); EC (saline > 4).
Values in the expected classes also confirm the units (% and kg/ha).
(4) Roadmap rule: a column missing for more than 50% of fields is not used as a model feature.

**Found:** 5 EC values of 0 were set to blank. Only 232 of 1,207 fields (19%) have all six lab values
(2022: 43, 2023: 60, 2024: 119, 2025: 10), in 52 villages. The values fit Latur's black cotton soils:
pH alkaline (99% above 7.0), potassium high (84% above 280 kg/ha), nitrogen and organic carbon mostly
low (51% and 54% below the low limit), EC normal for all fields. So the units (% and kg/ha) are right.
Lab values are missing for 81% of fields, so they are not used as a model feature for all fields.
They are kept for a subset check; soil for every field comes from SoilGrids rasters (Phase 3).

In [6]:
SOIL = ["soil_carbon", "soil_ph", "soil_ec", "soil_n", "soil_p", "soil_k"]

# (1) a value of exactly 0 is not a real measurement (EC of 0 is not possible in soil) -> blank
zeros = (f[SOIL] == 0)
print("single zeros set to blank:", {c: int(n) for c, n in zeros.sum().items() if n})
f[SOIL] = f[SOIL].mask(zeros)
f["flag_soil_zero_blanked"] = zeros.any(axis=1)

# (2) how many fields have lab values, and where
has_lab = f[SOIL].notna().all(axis=1)
f["has_soil_lab"] = has_lab
print(f"fields with all six lab values: {int(has_lab.sum())} of {len(f)} ({100 * has_lab.mean():.0f}%) "
      f"| by year: {f[has_lab].groupby('year').size().to_dict()} | villages: {f.loc[has_lab, 'village'].nunique()}")

# (3) are the values sensible? Compare with the Soil Health Card limits (ICAR-IISS):
#     organic carbon % (low < 0.5, high > 0.75); N, P, K kg/ha (low < 280 / 10 / 120, high > 560 / 25 / 280);
#     pH (acidic < 6.5, alkaline > 7.0); EC (saline > 4)
LIMITS = {"soil_carbon": (0.5, 0.75), "soil_n": (280, 560), "soil_p": (10, 25), "soil_k": (120, 280),
          "soil_ph": (6.5, 7.0), "soil_ec": (4.0, 4.0)}
rows = []
for c in SOIL:
    v = f.loc[has_lab, c]; lo, hi = LIMITS[c]
    rows.append({"column": c, "n": len(v), "min": v.min(), "median": v.median(), "max": v.max(),
                 "below_low_%": round(100 * (v < lo).mean()), "middle_%": round(100 * v.between(lo, hi).mean()),
                 "above_high_%": round(100 * (v > hi).mean())})
soil_tab = pd.DataFrame(rows).round(2)
soil_tab.to_csv(OUT / "R4_soil_lab.csv", index=False)
print(soil_tab.to_string(index=False))

# (4) rule from the roadmap: a column missing for more than 50% of fields is not used as a model feature
missing_pct = 100 * (1 - has_lab.mean())
print(f"\nmissing for {missing_pct:.0f}% of fields -> not a model feature for all fields; kept for a subset check "
      f"and a coverage map. Soil for every field comes from SoilGrids (Phase 3).")

single zeros set to blank: {'soil_ec': 5}
fields with all six lab values: 232 of 1207 (19%) | by year: {2022: 43, 2023: 60, 2024: 119, 2025: 10} | villages: 52
     column   n    min  median     max  below_low_%  middle_%  above_high_%
soil_carbon 232   0.02    0.48    1.10           54        31            15
    soil_ph 232   6.80    7.80    8.60            0         1            99
    soil_ec 232   0.02    0.30    1.18          100         0             0
     soil_n 232  62.72  273.00  672.00           51        43             6
     soil_p 232   7.10   14.36   43.50           16        69            16
     soil_k 232 162.57  554.48 1478.00            0        16            84

missing for 81% of fields -> not a model feature for all fields; kept for a subset check and a coverage map. Soil for every field comes from SoilGrids (Phase 3).


## Step 7: Is missing data related to yield?

For each column with gaps, we compare the yields of fields where it is missing with fields where it
is present (Mann-Whitney U, which uses ranks, so extreme yields do not drive it).
Because the gaps differ by year (for example, 2024 has no soil description and 2025 almost no lab values),
and yields also differ by year, the comparison is made within each year and the years are combined
(Fisher's method). Six tests are run at once, so the p-values are corrected (Holm, 1979).
Roadmap rule: where yield differs, a "missing" indicator is added as a feature.

**Found:** pooled over all years, missing soil description and missing soil lab values seem linked to
yield (p < 0.001). Within each year the link disappears (Fisher-combined p = 0.82 and 0.55; Holm-corrected
p = 1.0 for all six columns). The pooled result was a year effect: 2024 had high yields and no soil
description, 2025 lower yields and almost no lab values. No gap is related to yield, so no missing
indicators are added.

In [7]:
def holm(p):
    """Holm (1979) correction for several tests at once: adjusted p-values."""
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[order]).clip(max=1)
    out = np.empty(m); out[order] = adj
    return out

# what counts as "missing" for each column (after the fixes above)
MISSING = {
    "variety": f["variety"] == "Unknown",
    "sowing_date": f["sowing_date"].isna(),
    "harvest_date": f["harvest_date"].isna(),
    "soil_description": f["soil_color"].isna(),
    "soil_lab": ~f["has_soil_lab"],
    "farmer_area": f["area_actual_farmer"].isna() & (f["batch"] == "B"),   # only batch B was asked for it
}
y = f["crop_yield"]
rows = []
for name, miss in MISSING.items():
    # (a) all years together (Mann-Whitney U)
    p_all = stats.mannwhitneyu(y[miss], y[~miss]).pvalue
    # (b) within each year, then combined (Fisher's method), so a difference between years is not mistaken for one
    p_years = [stats.mannwhitneyu(y[miss & (f["year"] == yr)], y[~miss & (f["year"] == yr)]).pvalue
               for yr in sorted(f["year"].unique())
               if (miss & (f["year"] == yr)).sum() >= 5 and (~miss & (f["year"] == yr)).sum() >= 5]
    p_within = stats.combine_pvalues(p_years, method="fisher").pvalue if p_years else np.nan
    rows.append({"column": name, "missing": int(miss.sum()), "missing_%": round(100 * miss.mean(), 1),
                 **{str(yr): int((miss & (f["year"] == yr)).sum()) for yr in sorted(f["year"].unique())},
                 "median_yield_missing": y[miss].median(), "median_yield_present": y[~miss].median(),
                 "p_all_years": p_all, "years_tested": len(p_years), "p_within_years": p_within})
miss_tab = pd.DataFrame(rows)
ok = miss_tab["p_within_years"].notna()
miss_tab.loc[ok, "p_within_holm"] = holm(miss_tab.loc[ok, "p_within_years"])   # six tests at once
miss_tab["yield_differs"] = miss_tab["p_within_holm"] < 0.05
miss_tab.round(4).to_csv(OUT / "R4_missingness.csv", index=False)
print(miss_tab.round(3).to_string(index=False))

# rule from the roadmap: where yield differs, keep a "missing" indicator as a feature
added = []
for name in miss_tab.loc[miss_tab["yield_differs"], "column"]:
    f[f"missing_{name}"] = MISSING[name]; added.append(f"missing_{name}")
print("\nmissing indicators added:", added or "none needed")

          column  missing  missing_%  2022  2023  2024  2025  median_yield_missing  median_yield_present  p_all_years  years_tested  p_within_years  p_within_holm  yield_differs
         variety       45        3.7    20     7    18     0                   9.0                 8.085        0.125             3           0.260            1.0          False
     sowing_date       39        3.2    17     2    17     3                   9.0                 8.155        0.267             2           0.324            1.0          False
    harvest_date       22        1.8     2     3     6    11                   8.0                 8.220        0.840             2           0.335            1.0          False
soil_description      233       19.3    29     0   204     0                   9.0                 8.000        0.000             2           0.823            1.0          False
        soil_lab      975       80.8    12     0   204   759                   8.0                 9.000      

## Step 8: All R4 flags in one table, and save

Each check left a True/False flag. "Fixed" flags mark values we corrected with a tested rule
(year typos, swapped harvest dates, variety typos, zero soil values). "Problem" flags mark values
that are still doubtful (missing, implausible, unusual, undecided). Nothing is removed;
R5 decides what to do with each flag. Saved as `data/processed_v2/R4_attributes.gpkg`.

**Found:** 111 fields had at least one value fixed: 85 swapped harvest dates (79 in 2025, none in 2022),
15 sowing years, 3 harvest years, 4 variety typos, 5 zero soil values. 199 fields have at least one
doubtful value: harvested before any variety matures (91), unknown variety (45), sowing unusual for
its year (43), no sowing date (34), harvest date undecided (22), unusual duration (6), implausible
sowing (5). Note: R1 suggested many 2022 harvest dates were swapped; this step shows the 2022
November dates have no possible swapped reading, so they are real early-November harvests.

In [8]:
f["flag_variety_typo"] = f["variety_match"] == "one-digit typo"
f["flag_variety_unknown"] = f["variety"] == "Unknown"

R4_FLAGS = ["flag_sowing_year_fixed", "flag_sowing_implausible", "flag_sowing_missing", "flag_sowing_unusual",
            "flag_harvest_year_fixed", "flag_harvest_swapped", "flag_harvest_undecided",
            "flag_duration_unusual", "flag_harvest_before_maturity",
            "flag_variety_typo", "flag_variety_unknown", "flag_soil_zero_blanked"]
FIXED = ["flag_sowing_year_fixed", "flag_harvest_year_fixed", "flag_harvest_swapped", "flag_variety_typo", "flag_soil_zero_blanked"]
for c in R4_FLAGS:
    f[c] = f[c].fillna(False).astype(bool)

summary = pd.DataFrame({"type": ["fixed" if c in FIXED else "problem" for c in R4_FLAGS],
                        "fields": [int(f[c].sum()) for c in R4_FLAGS],
                        **{str(y): [int(f.loc[f["year"] == y, c].sum()) for c in R4_FLAGS] for y in sorted(f["year"].unique())}},
                       index=R4_FLAGS)
summary.to_csv(OUT / "R4_flag_summary.csv")
print(summary.to_string())
PROBLEMS = [c for c in R4_FLAGS if c not in FIXED]
print("\nfields with at least one attribute problem flag:", int(f[PROBLEMS].any(axis=1).sum()), "of", len(f),
      "| fields with at least one fix:", int(f[FIXED].any(axis=1).sum()))

out = DATA2 / "R4_attributes.gpkg"
f.to_file(out, driver="GPKG")
print("saved:", out, "| columns:", f.shape[1])

                                 type  fields  2022  2023  2024  2025
flag_sowing_year_fixed          fixed      15     0     0     3    12
flag_sowing_implausible       problem       5     0     2     0     3
flag_sowing_missing           problem      34    17     0    17     0
flag_sowing_unusual           problem      43     0     3     4    36
flag_harvest_year_fixed         fixed       3     1     0     2     0
flag_harvest_swapped            fixed      85     0     0     6    79
flag_harvest_undecided        problem      22     2     3     6    11
flag_duration_unusual         problem       6     0     2     3     1
flag_harvest_before_maturity  problem      91     4     9    28    50
flag_variety_typo               fixed       4     1     0     3     0
flag_variety_unknown          problem      45    20     7    18     0
flag_soil_zero_blanked          fixed       5     5     0     0     0

fields with at least one attribute problem flag: 199 of 1207 | fields with at least one f

## Step 9: Sowing and harvest dates by year (figure)

Box plots of the sowing and harvest day of each year, after the fixes above. The box is Q1 to Q3,
and the whiskers reach the same 3 IQR fences used in Steps 1 and 4. Red dots are flagged fields:
sowing unusual for its year (left) and harvest before any variety matures (right).
Saved in `reports/figures/phase1a/`.

**Found:**

In [ ]:
import matplotlib.pyplot as plt

FIG = ROOT / "reports" / "figures" / "phase1a"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

YEARS = sorted(f["year"].unique())
panels = [
    ("Sowing date", f["sowing_doy"], f["flag_sowing_unusual"], "Sowing unusual for its year"),
    ("Harvest date (after the swap fix)", f["harvest_date"].dt.dayofyear, f["flag_harvest_before_maturity"],
     "Harvested before any variety matures"),
]
# month starts as day of year (non-leap year; 2024 is shifted by one day, too small to matter)
MONTHS = {"1 Jun": 152, "1 Jul": 182, "1 Aug": 213, "1 Sep": 244, "1 Oct": 274, "1 Nov": 305, "1 Dec": 335}
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (title, values, flag, flag_label) in zip(axes, panels):
    data = [values[(f["year"] == y) & values.notna()] for y in YEARS]
    ax.boxplot(data, vert=False, widths=0.5, whis=3, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="#E8EDF6", edgecolor="#1F3864"),
               medianprops=dict(color="#1F3864", lw=1.6),
               whiskerprops=dict(color="#1F3864"), capprops=dict(color="#1F3864"))
    for i, d in enumerate(data, start=1):
        is_flag = flag.loc[d.index].to_numpy(dtype=bool)
        yj = i + rng.uniform(-0.18, 0.18, len(d))
        ax.scatter(d[~is_flag], yj[~is_flag], s=6, color="#1F3864", alpha=0.35, lw=0)
        ax.scatter(d[is_flag], yj[is_flag], s=14, color="#C00000", alpha=0.9, lw=0,
                   label=flag_label if i == 1 else None)
    lo, hi = min(d.min() for d in data), max(d.max() for d in data)
    ax.set_xticks(list(MONTHS.values()), list(MONTHS.keys()))
    ax.set_xlim(lo - 5, hi + 5)
    ax.set_title(title, loc="left", fontsize=12)
    ax.grid(axis="x", color="#E5E5E5", lw=0.6); ax.set_axisbelow(True)
    ax.legend(loc="lower right", frameon=False, fontsize=10)
axes[0].set_yticks(range(1, len(YEARS) + 1), [str(y) for y in YEARS])
fig.tight_layout()
fig.savefig(FIG / "R4_sowing_harvest_by_year.png", dpi=300, facecolor="white")
plt.show()

# the numbers behind the figure
day = lambda y, n: (pd.Timestamp(int(y), 1, 1) + pd.Timedelta(days=int(n) - 1)).strftime("%d %b")
rows = []
for y in YEARS:
    s = f.loc[f["year"] == y, "sowing_doy"].dropna()
    h = f.loc[f["year"] == y, "harvest_date"].dt.dayofyear.dropna()
    rows.append({"year": y, "sowing_n": len(s), "sowing_median": day(y, s.median()),
                 "harvest_n": len(h), "harvest_median": day(y, h.median()),
                 "sowing_unusual": int(f.loc[f["year"] == y, "flag_sowing_unusual"].sum()),
                 "harvest_before_maturity": int(f.loc[f["year"] == y, "flag_harvest_before_maturity"].sum())})
print(pd.DataFrame(rows).to_string(index=False))

## Summary of R4

**What R4 did:** checked and fixed the non-spatial values. Every rule comes from a statistical
test or a published source. Output: `data/processed_v2/R4_attributes.gpkg`.

1. **Sowing dates:** window 1 June to 5 August (Maharashtra field trials). 15 year typos fixed,
   5 implausible, 34 missing. Sowing differs by year (Kruskal-Wallis p < 0.001), so each year is
   checked against its own range: 43 unusual. The fixed dates are consistent with their year
   (Mann-Whitney p = 0.30), a weak check.
2. **Harvest dates:** the day/month swap is real (chi-square p < 0.001). Each unclear date takes the
   reading whose crop duration is at least 3 times as likely (Kass and Raftery): 85 swapped,
   409 kept, 22 undecided. The fix is partial: November and odd months now match the expected season,
   but September stays too high because of 61 early harvests, which are not swaps.
3. **Varieties:** 73 spellings become 11 varieties by code number; 4 one-digit typos, 45 Unknown.
4. **Crop duration:** median 114 days. It differs by year (p < 0.001), not by variety (p = 0.07).
   91 fields were harvested before 91 days, earlier than any listed variety matures (ICAR).
5. **Categories:** groups under 10 fields merged (Cochran's rule); "Slit" corrected to "Silt".
6. **Soil lab values:** fit Latur's black soils (Soil Health Card limits, ICAR-IISS), but only
   232 fields (19%) have them, so they are not a model feature; SoilGrids is used in Phase 3.
7. **Missing data:** not related to yield within years (Mann-Whitney, Fisher, Holm); the pooled
   link was a year effect. No missing indicators needed.
8. **Flags:** 111 fields fixed, 199 with at least one doubtful value. Nothing removed; R5 decides.
9. **Figure:** sowing and harvest dates by year, with the flagged fields marked.

**Limits:**
- All tests here treat each field as independent. Fields of the same farmer, village or plot are
  related, and related data makes p-values too small. The strong results (p < 0.001) are far from
  the 0.05 line and still stand. The "no difference" results stay non-significant, because
  relatedness can only make a difference look stronger. How strongly fields are related in space
  is measured in Phase 2 (Moran's I, semivariogram).
- The 91-day maturity limit comes from the earliest listed variety (MAUS 612). Maturity for
  KDS 753 and DS 228 was not found, so this is the lenient choice.
- Cochran's rule (at least 10 fields per group) is a common guideline, not a strict law.

**Saved in `results/phase1a/`:** R4_sowing_by_year, R4_variety_lookup, R4_duration_by_year,
R4_categories, R4_soil_lab, R4_missingness, R4_flag_summary.
**Figure:** `reports/figures/phase1a/R4_sowing_harvest_by_year.png`.